## STAC/ZARR 2025 workshop

This notebook shows the following features:
- CADIP STAC search
- AUXIP STAC search
- Staging of S1A CADIP session as a STAC item
- Staging of S1A auxiliary files (MPL_ORBPRE / MPL_ORBSCT) as STAC items

## 1. Initialization

In [1]:
# Init environment before running a demo notebook.
import os
from resources.utils import *  
from resources.dask_utils import *

init_demo()
os.environ["JUPYTERHUB_API_TOKEN"] = "xxx"
init_dask_cluster_staging(scale=4)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

# You can check here the number of workers, threads and memory per worker.
display(dask_cluster_staging)

DependencyConflict: requested: "starlette >= 0.13, <0.15" but found: "starlette 0.46.2"


Auxip service: http://rs-server-adgs:8000/auxip
CADIP service: http://rs-server-cadip:8000/cadip
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000
DPR service: http://rs-dpr-service:8000
Connecting to dask gateway for 'dask-staging': http://dask-staging:8000 ...
image = dfe2803ab17d449ca99eb89857d41328
Get existing dask cluster: 'dfe2803ab17d449ca99eb89857d41328'
Dask dashboard for 'dask-staging': http://localhost:8701/clusters/dfe2803ab17d449ca99eb89857d41328/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.5  | 2.2.4     | 2.2.4   |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask workers for 'dask-staging' are up: 2/4
Dask workers for 'dask-staging' are up: 4/4


## 1. Find S1A CADIP session from Matera as a STAC item

In [2]:
# Retrieve https://stac-browser-cadip.ops.rs-python.eu/collections/s1_mti/items/S1A_20250413151551058740
# Here we show support of cql2-text filters in GET /search
cadip_session = cadip_client.search(method="GET", limit=1, collections="s1_mti",
                                    stac_filter="id=S1A_20250413151551058740",
                                    sortby=[ { "field": "start_datetime", "direction": "desc" } ])[0]
cadip_session_href = cadip_session.get_links("self")[0].get_href()
print(f"CADIP session: {cadip_session_href}")

IndexError: list index out of range

## 2. Find auxiliary data from AUXIP as STAC items

In [ ]:
# Retrieve https://stac-browser-auxip.ops.rs-python.eu/collections/S1-MPL_ORBPRE/items/S1A_OPER_MPL_ORBPRE_20250412T021831_20250419T021830_0001.EOF.zip
# Here we show support of cql2-json filters in POST /search
mpl_orbpre = auxip_client.search(method="POST", limit=1, collections="S1-MPL_ORBPRE", stac_filter={
        "op": "and",
        "args": [
            { "op": "=", "args": [ { "property": "product:type" }, "MPL_ORBPRE" ] },
            { "op": "=", "args": [ { "property": "published" }, "2025-04-12T02:47:24.210000Z" ] }
        ]
    }, sortby=[ { "field": "start_datetime", "direction": "desc" } ])[0]
mpl_orbpre_href = mpl_orbpre.get_links("self")[0].get_href()
print(f"MPL_ORBPRE: {mpl_orbpre_href}")

# Retrieve https://stac-browser-auxip.ops.rs-python.eu/collections/S1-MPL_ORBSCT/items/S1A_OPER_MPL_ORBSCT_20140507T150704_99999999T999999_0027.EOF.zip
mpl_orbsct = auxip_client.search(method="POST", limit=1, collections="S1-MPL_ORBSCT", stac_filter={
        "op": "and",
        "args": [
            { "op": "=", "args": [ { "property": "product:type" }, "MPL_ORBSCT" ] },
            { "op": "=", "args": [ { "property": "published" }, "2025-01-29T07:57:16.994000Z" ] }
        ]
    }, sortby=[ { "field": "start_datetime", "direction": "desc" } ])[0]
mpl_orbsct_href = mpl_orbsct.get_links("self")[0].get_href()
print(f"MPL_ORBSCT: {mpl_orbsct_href}")

## 3. Create Catalog collections

In [ ]:
temporal = TemporalExtent([datetime(2014, 4, 3), datetime(2025, 4, 17)])

s1_sessions_coll = get_or_create_test_collection(
    collection_id="s1_sessions", description="S1 staged CADIP sessions", title="S1 sessions", temporal=temporal)
mpl_orbpres_coll = get_or_create_test_collection(
    collection_id="s1_aux_mpl_orbpre", description="MPL_ORBPRE staged auxiliary files", title="MPL_ORBPRE", temporal=temporal)
mpl_orbscts_coll = get_or_create_test_collection(
    collection_id="s1_aux_mpl_orbsct", description="MPL_ORBSCT staged auxiliary files", title="MPL_ORBSCT", temporal=temporal)

print(f"S1 session collection: {s1_sessions_coll.get_links('self')[0].get_href()}")
print(f"MPL_ORBPRE collection: {mpl_orbpres_coll.get_links('self')[0].get_href()}")
print(f"MPL_ORBSCT collection: {mpl_orbscts_coll.get_links('self')[0].get_href()}")
print("STAC Browser: https://stac-browser-catalog.ops.rs-python.eu")

## 4. Stage session and auxiliary data in the STAC Catalog

In [ ]:
stage_single_item(mpl_orbpre, mpl_orbpres_coll)
stage_single_item(mpl_orbsct, mpl_orbscts_coll)
stage_single_item(cadip_session, s1_sessions_coll, timeout = 3600)